# Trace Generation

This notebook generates the simulated power traces used in the attack:

1. Load (or generate) the secret key `f`.
2. Sample hashes `c` and filter them by the ratio |Re(C)| / |Im(C)| of their FFT coefficients `C = FFT(c)`.
3. For each target index, write the inputs `FFT(f)[idx]` and `FFT(c)[idx]` for ELMO, run ELMO, check its output against Python, and save the truncated traces.

## Prerequisites

Before generating traces, build the target binary `ELMO/FPR/FprMul.bin` by running `make` in the `ELMO/FPR/` directory. Check the NOTRACES in FprMul.c:

```
cd ELMO/FPR
make
```

# Setup

In [ ]:
%matplotlib ipympl
import gc
import glob
import os
import subprocess
import time

import matplotlib.pyplot as plt
import numpy as np
from tqdm import notebook

from elmo_util import generate_elmo_plaintexts, load_combined_hex, load_elmo_traces
from falcon_gen import gen_c, gen_f
from fft_util import FFT

## Parameters

In [ ]:
size_f = 512        # Ring degree n
B = 11              # Bound on |f1 / f2|, reported for each target index
D = B * 64          # Filtering threshold on |c1 / c2|

fft = FFT(size_f)

# Small indices (k % 64 in {0, 63}) rarely pass the two-sided filter,
# so they are filtered on one side only.
SMALL_INDICES = [k for k in range(size_f // 2) if k % 64 in (0, 63)]
REGULAR_INDICES = [k for k in range(size_f // 2) if k not in SMALL_INDICES]

# Filtering sides: |re| > D x |im| or |im| > D x |re|.
SIDES = ("re_Dx_im", "im_Dx_re")

# Files and directories
KEY_FILE = "f.npy"
HASH_DIR = "hashes"
UNFILTERED_HASH_FILE = os.path.join(HASH_DIR, "c_list.npy")
RAW_TRACE_DIR = "raw_traces"
TRUNC_TRACE_DIR = "truncated_traces"
PLAINTEXT_FILE = "plaintexts.txt"
ELMO_DIR = "ELMO"
ELMO_BINARY = "FPR/FprMul.bin"
ELMO_PRINTDATA = os.path.join(ELMO_DIR, "output", "printdata.txt")
ELMO_TRACE_DIR = os.path.join(ELMO_DIR, "output", "traces")

# ELMO traces of our binary have length 356199. We keep only the last 99 samples to save space.
TRUNC_START = 356100

# What to do when a trace file already exists in the batch loops: "skip", "overwrite" or "ask".
TRACE_IF_EXISTS = "skip"

# Pause between ELMO runs to avoid writing large amounts of data to disk in a short time.
COOLDOWN_SECONDS = 5

## Helper Functions

In [ ]:
def check_side(side):
    """Raise a ValueError if side is not a valid filtering side."""
    if side not in SIDES:
        raise ValueError(f"side must be one of {SIDES}, got {side!r}")


def side_tag(side):
    """Return the file-name tag of a filtering side, e.g. 're_704x_im' for 're_Dx_im'."""
    check_side(side)
    return side.replace("Dx", f"{D}x")


def describe_side(side):
    """Return a readable description of a filtering side, e.g. '|re| > 704 x |im|'."""
    check_side(side)
    return f"|re| > {D} x |im|" if side == "re_Dx_im" else f"|im| > {D} x |re|"


def hash_path(side, target_idx):
    """Return the path of the filtered hash file for a side and target index."""
    return os.path.join(HASH_DIR, f"c_{side_tag(side)}_idx{target_idx}.npy")


def trace_path(trace_dir, side, target_idx):
    """Return the path of a trace file. Use side=None for traces of unfiltered hashes."""
    if side is None:
        return os.path.join(trace_dir, f"traces_idx{target_idx}.npy")
    return os.path.join(trace_dir, f"traces_{side_tag(side)}_idx{target_idx}.npy")


def filter_mask(C, side):
    """Return whether the FFT coefficient(s) C pass the filter on the given side."""
    check_side(side)
    ratio = np.abs(C.real / C.imag)
    return ratio > D if side == "re_Dx_im" else ratio < 1 / D


def should_write(filepath, if_exists):
    """Decide whether to write filepath when it may already exist.

    Args:
        filepath (str): Path of the file to write.
        if_exists (str): "skip", "overwrite" or "ask" (prompt the user).

    Returns:
        bool: True if the file should be written.
    """
    if not os.path.exists(filepath) or if_exists == "overwrite":
        return True
    if if_exists == "ask":
        response = input(f"File '{filepath}' already exists. Overwrite? [y/N]: ").strip().lower()
        if response == "y":
            return True
        print("Save cancelled.")
        return False
    print(f"File '{filepath}' already exists. Skipped.")
    return False


def save_array(filepath, array, if_exists="ask"):
    """Save an array as .npy, creating the parent directory if needed."""
    if not should_write(filepath, if_exists):
        return
    array = np.asarray(array)
    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)
    np.save(filepath, array)
    print(f"Saved array of shape {array.shape} to {filepath}")


def gen_c_list_one(n_hashes, target_idx, side):
    """Sample hashes until n_hashes of them pass the filter on one side for target_idx."""
    check_side(side)
    c_list, n_sampled = [], 0
    with notebook.tqdm(total=n_hashes) as pbar:
        while len(c_list) < n_hashes:
            c = gen_c(size_f)
            n_sampled += 1
            if filter_mask(fft.forward(c)[target_idx], side):
                c_list.append(c)
                pbar.update(1)
    print(f"Collected {n_hashes} hashes with {describe_side(side)} for idx {target_idx} "
          f"out of {n_sampled} sampled")
    return c_list


def gen_c_list_all(n_hashes):
    """Sample n_hashes hashes without filtering."""
    c_list = [gen_c(size_f) for _ in notebook.trange(n_hashes)]
    print(f"Collected {n_hashes} hashes")
    return c_list


def get_c_components(c_array, target_idx):
    """Return the real and imaginary parts of FFT(c)[target_idx] for each hash c."""
    C_k = np.array([fft.forward(c)[target_idx] for c in c_array])
    return list(C_k.real), list(C_k.imag)


def report_f_ratio(target_idx):
    """Print |f1 / f2| for the target index and whether it lies within [1/B, B]."""
    f1, f2 = F[target_idx].real, F[target_idx].imag
    ratio = np.abs(f1 / f2)
    within = 1 / B <= ratio <= B
    print(f"|f1/f2| = {ratio} is {'' if within else 'NOT '}within 1/{B} and {B}")


def run_elmo():
    """Run ELMO on PLAINTEXT_FILE. Raise a RuntimeError if it fails."""
    binary_path = os.path.join(ELMO_DIR, ELMO_BINARY)
    if not os.path.exists(binary_path):
        raise FileNotFoundError(f"{binary_path} not found. Build it by running make in {os.path.dirname(binary_path)}/")
    print("Running ELMO...")
    result = subprocess.run(["./elmo", ELMO_BINARY, "-nofvr"],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, cwd=ELMO_DIR)
    if result.returncode != 0:
        raise RuntimeError(f"ELMO failed with return code {result.returncode}")
    print("ELMO finished")


def print_float(name, x):
    """Print a float64 as its bit pattern (as an integer) and its value."""
    print(f"{name} = {np.float64(x).view(np.uint64)} ({x})")


def check_elmo_output(f1, f2, c1_list, c2_list, c_idx=0):
    """Check that ELMO and Python compute the same Re(f * c), bit for bit, for every hash.

    Prints the inputs and results for hash c_idx as an example.
    Raises a RuntimeError if any result differs.
    """
    res_elmo = load_combined_hex(ELMO_PRINTDATA)
    res_python = np.float64(f1) * np.array(c1_list) - np.float64(f2) * np.array(c2_list)
    if len(res_elmo) != len(res_python):
        raise RuntimeError(f"ELMO returned {len(res_elmo)} results for {len(res_python)} hashes")

    print_float("f_re", f1)
    print_float("f_im", f2)
    print_float("c_re", c1_list[c_idx])
    print_float("c_im", c2_list[c_idx])
    print_float("Python res_re", res_python[c_idx])
    print_float("ELMO   res_re", res_elmo.view(np.float64)[c_idx])

    n_mismatch = np.count_nonzero(res_python.view(np.uint64) != res_elmo)
    if n_mismatch:
        raise RuntimeError(f"{n_mismatch} of {len(res_elmo)} ELMO results differ from Python")
    print(f"All {len(res_elmo)} ELMO results match Python")


def delete_elmo_traces():
    """Delete the raw .trc files written by ELMO."""
    files = glob.glob(os.path.join(ELMO_TRACE_DIR, "*.trc"))
    for path in files:
        os.remove(path)
    print(f"Deleted {len(files)} .trc file(s)")


def generate_traces(target_idx, c_array, filepath, if_exists=TRACE_IF_EXISTS):
    """Run ELMO on f * c for every hash c at target_idx, check the output, and save truncated traces."""
    if not should_write(filepath, if_exists):
        return
    time.sleep(COOLDOWN_SECONDS)

    f1, f2 = F[target_idx].real, F[target_idx].imag
    c1_list, c2_list = get_c_components(c_array, target_idx)
    generate_elmo_plaintexts([f1] + c1_list, [f2] + c2_list, PLAINTEXT_FILE)
    print(f"Wrote {len(c1_list)} hashes to {PLAINTEXT_FILE}")

    run_elmo()
    check_elmo_output(f1, f2, c1_list, c2_list)

    traces = load_elmo_traces(ELMO_TRACE_DIR)
    trunc_traces = traces[:, TRUNC_START:]
    print(f"Truncated traces from {traces.shape} to {trunc_traces.shape}")
    del traces
    save_array(filepath, trunc_traces, if_exists="overwrite")
    del trunc_traces
    gc.collect()


def plot_trace(trace):
    """Plot a single trace."""
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(trace, linewidth=0.7, color="tab:blue")
    ax.set_xlabel("Cycle number")
    ax.set_ylabel("Leakage value")
    ax.grid(True, alpha=0.3)
    plt.show()

# Load Key

In [ ]:
# Set to True to sample a new key and overwrite KEY_FILE.
# Traces generated with the old key will no longer match.
GENERATE_NEW_KEY = False

if GENERATE_NEW_KEY or not os.path.exists(KEY_FILE):
    f = gen_f(size_f)
    np.save(KEY_FILE, f)
    print(f"Generated a new key and saved it to {KEY_FILE}")
else:
    f = np.load(KEY_FILE)
    print(f"Loaded the key from {KEY_FILE}")
F = fft.forward(f)

# Generate Hashes

Every target index gets 200 hashes in total: 100 per side for regular indices, and 200 on the `im_Dx_re` side for small indices.

## Two-Sided Filtering (Regular Indices)

In [ ]:
n_hashes = 100
half = size_f // 2

# c_lists[side][k] holds the hashes that pass the filter on that side for index k.
c_lists = {side: [[] for _ in range(half)] for side in SIDES}
n_sampled, current_min = 0, 0

with notebook.tqdm(total=n_hashes) as pbar:
    while current_min < n_hashes:
        c = gen_c(size_f)
        C = fft.forward(c)
        n_sampled += 1
        for side in SIDES:
            for k in np.flatnonzero(filter_mask(C[:half], side)):
                if len(c_lists[side][k]) < n_hashes:
                    c_lists[side][k].append(c)
        # Progress is limited by the regular index with the fewest hashes.
        current_min = min(len(c_lists[side][k]) for side in SIDES for k in REGULAR_INDICES)
        pbar.update(current_min - pbar.n)

print(f"Sampled {n_sampled} hashes")
for k in REGULAR_INDICES:
    for side in SIDES:
        save_array(hash_path(side, k), c_lists[side][k], if_exists="overwrite")

## One-Sided Filtering (Small Indices)

In [ ]:
n_hashes = 200
for k in SMALL_INDICES:
    c_list = gen_c_list_one(n_hashes, k, "im_Dx_re")
    save_array(hash_path("im_Dx_re", k), c_list, if_exists="overwrite")
    print()

## No Filtering

In [ ]:
n_hashes = 200
c_list = gen_c_list_all(n_hashes)
save_array(UNFILTERED_HASH_FILE, c_list, if_exists="overwrite")

# ELMO

## Test on a Single Index

Before running the batch loops below, check the pipeline step by step on one index.
Make sure you have built `ELMO/FPR/FprMul.bin` first (see Prerequisites).

### Generate ELMO Inputs

In [ ]:
target_idx = 0
side = "im_Dx_re"

c_array = np.load(hash_path(side, target_idx))
print(f"Loaded {len(c_array)} hashes with {describe_side(side)} for idx {target_idx}")

f1, f2 = F[target_idx].real, F[target_idx].imag
c1_list, c2_list = get_c_components(c_array, target_idx)
generate_elmo_plaintexts([f1] + c1_list, [f2] + c2_list, PLAINTEXT_FILE)
print(f"Wrote {len(c1_list)} hashes to {PLAINTEXT_FILE}")

### Run ELMO and Check the Output

This runs `./elmo FPR/FprMul.bin -nofvr` in the `ELMO/` directory, then checks that ELMO and Python produce bit-identical results.

In [ ]:
run_elmo()
check_elmo_output(f1, f2, c1_list, c2_list)

### Load and Save Traces

In [ ]:
traces = load_elmo_traces(ELMO_TRACE_DIR)
print(f"Loaded {traces.shape[0]} traces of length {traces.shape[1]}")
save_array(trace_path(RAW_TRACE_DIR, side, target_idx), traces, if_exists="ask")

trunc_traces = traces[:, TRUNC_START:]
print(f"Truncated traces from {traces.shape} to {trunc_traces.shape}")
save_array(trace_path(TRUNC_TRACE_DIR, side, target_idx), trunc_traces, if_exists="ask")

In [ ]:
plot_trace(trunc_traces[0])

## Generate Traces with Filtering

In [ ]:
# Indices to generate traces, e.g., SMALL_INDICES, REGULAR_INDICES, range(10)
# Ensure that the number of hashes matches the number of traces in FprMul.c
TARGET_INDICES = REGULAR_INDICES

try:
    for target_idx in TARGET_INDICES:
        print("-" * 85)
        print(f"Parameters: B = {B}, D = {D}, target_idx = {target_idx}")
        report_f_ratio(target_idx)
        sides = ["im_Dx_re"] if target_idx in SMALL_INDICES else SIDES
        for side in sides:
            print(f"\nLoading hashes from {hash_path(side, target_idx)}")
            c_array = np.load(hash_path(side, target_idx))
            generate_traces(target_idx, c_array, trace_path(TRUNC_TRACE_DIR, side, target_idx))
finally:
    delete_elmo_traces()

## Generate Traces without Filtering

In [ ]:
# Indices to generate traces, e.g. range(10) for a quick test.
TARGET_INDICES = range(size_f // 2)
c_list = np.load(UNFILTERED_HASH_FILE)

try:
    for target_idx in TARGET_INDICES:
        print("-" * 85)
        print(f"Parameters: B = {B}, target_idx = {target_idx}")
        report_f_ratio(target_idx)
        generate_traces(target_idx, c_list, trace_path(TRUNC_TRACE_DIR, None, target_idx))
finally:
    delete_elmo_traces()